In [ ]:
%matplotlib notebook
from rfsoc_rfdc.rfsoc_overlay import RFSoCOverlay
from rfsoc_rfdc.overlay_task import OverlayTask
from rfsoc_rfdc.overlay_task import BlinkLedTask

from rfsoc_rfdc.transmitter.single_ch_tx_task import SingleChTxTask
from rfsoc_rfdc.receiver.single_ch_rx_task import SingleChRxTask
from rfsoc_rfdc.beamformer_task import BeamformerTxTask

from rfsoc_rfdc.transmitter.multi_ch_tx_indept_task import MultiChTxIndeptTask
from rfsoc_rfdc.receiver.multi_ch_rx_indept_task import MultiChRxIndeptTask

from rfsoc_rfdc.rfdc_task import RfdcTask 
from rfsoc_rfdc.mts_task import MtsTask
from rfsoc_rfdc.array_calib_task import ArrayCalibTask
from rfsoc_rfdc.overlay_task import OverlayTask, TASK_STATE

from rfsoc_rfdc.rfdc_config import ZCU216_CONFIG

import sys
import os
import time

In [ ]:
from rfsoc_rfdc.dsp.ofdm import OFDM
from rfsoc_rfdc.dsp.detection import Detection

In [ ]:
ol = RFSoCOverlay(path_to_bitstream="./rfsoc_rfdc/bitstream/rfsoc_rfdc_v47_8t1r_bf.bit")
NEW_CONFIG = {
    "RefClockForPLL": 300.0,
    "DACSampleRate": 2400.0,
    "DACInterpolationRate": 4,
    "DACNCO": 700,
    "ADCSampleRate": 2400.0,
    "ADCInterpolationRate": 4,
    "ADCNCO": -700
}
ZCU216_CONFIG.update(NEW_CONFIG)

In [ ]:
rfdc_t = RfdcTask(ol, debug_mode=True, board="ZCU216")
mts_t = MtsTask(ol, board="ZCU216", debug_mode=True)

for task in [mts_t, rfdc_t]:
    task.start()
    task.join()

In [ ]:
calib_task = ArrayCalibTask(ol, num_dacs=8, num_adcs= 1)
calib_t.start()
calib_t.join()

In [ ]:
true_samp_rate = ZCU216_CONFIG['DACSampleRate'] / ZCU216_CONFIG['DACInterpolationRate'] * 1e6

bf_task = BeamformerTxTask(ol, debug_mode=False, num_channels=8)
bf_task.calib_steer(0)

In [ ]:
# EVM, BER vs SNR experiment
led_t = BlinkLedTask(ol)
led_t.start()

for qam_order in ["QPSK", "16QAM", "64QAM", "256QAM"]:
    for atten in range(0, 31, 6):

        ZCU216_CONFIG['CONFIG_NAME'] = "CHARM_OTA_8T1R_" + qam_order + "_" + str(atten)
        ZCU216_CONFIG['OFDM_ATTEN_DB'] = atten
        ZCU216_CONFIG['QAM'] = qam_order
        ZCU216_CONFIG['OFDM_SCHEME'] = OFDM(sym_num=100, fft_size=1024, sub_num=768, modu=qam_order, cp_rate=1/14)
        ZCU216_CONFIG['DETECTION_SCHEME'] = Detection(sample_rate=true_samp_rate)

        print(f"QAM={qam_order}, OFDM_ATTEN_DB={atten}")

        tx_t = SingleChTxTask(ol, mode="iq2real")
        tx_t.start()
        
        rx_t = SingleChRxTask(ol, mode="real2iq")
        rx_t.start()
        
        while rx_t.task_state != TASK_STATE["STOP"]:
            time.sleep(1)
        
        tx_t.stop()
        rx_t.stop()

led_t.stop()